# Dataset B — Final six-model review

## tl;dr

Notebook này là scaffold chỉ đọc cho so sánh cuối LR, RF, XGBoost, PlainMLP, RealMLP và TabMmini. Nó chỉ mở cross-model results khi cả sáu reporting summaries đều `COMPLETE`, dùng final 15-feature contract, cùng `split_v1`, cùng ordered Test IDs và cùng bootstrap-index lineage. Hiện tại notebook phải hiển thị readiness, không trộn kết quả historical.

## Context & Methods

Mỗi final model run sẽ xuất reporting package chuẩn gồm `model_summary.json`, `cv_fold_metrics.csv`, `rq1_metrics_long.csv`, `confusion_matrix.csv`, `calibration_curve.csv` và `artifact_index.json`. Notebook tổng hợp không huấn luyện, inference, chọn threshold hay tính lại bootstrap.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'config/dataset_b/model_protocol_v1.yaml').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Cannot locate the Dataset B project root.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.dataset_b.reporting import EXPECTED_REPORTING_FILES, FINAL_MODEL_KEYS, validate_cross_model_summaries, validate_reporting_directory

MODEL_SUMMARY_PATHS = {model: None for model in FINAL_MODEL_KEYS}
MODEL_SUMMARY_PATHS['rf'] = 'artifacts/dataset_b/rf/final/rf_final_15f_20261002_01/reporting/model_summary.json'
MODEL_SUMMARY_PATHS['xgboost'] = 'artifacts/dataset_b/xgboost/final/xgb_final_15f_20261003_01/reporting/model_summary.json'
MODEL_SUMMARY_PATHS['plain_mlp'] = 'artifacts/dataset_b/plain_mlp/final/plain_mlp_final_15f_20261004_03/reporting/model_summary.json'
print('Reporting contract files:', ', '.join(EXPECTED_REPORTING_FILES))

Reporting contract files: model_summary.json, cv_fold_metrics.csv, rq1_metrics_long.csv, confusion_matrix.csv, calibration_curve.csv, artifact_index.json


### Key Assumptions

- Chỉ đường dẫn tới `final/<run_id>/reporting/model_summary.json` được phép điền.
- Working, legacy 12-feature và incomplete summaries không đủ điều kiện so sánh.
- CI riêng từng model không tự động chứng minh chênh lệch giữa model có ý nghĩa thống kê.

## Data

### Reporting readiness

In [2]:
summaries = []
readiness_rows = []
for model_key in FINAL_MODEL_KEYS:
    configured = MODEL_SUMMARY_PATHS[model_key]
    if configured is None:
        readiness_rows.append({'model_key': model_key, 'reporting_summary': 'NOT_CONFIGURED', 'status': 'PENDING'})
        continue
    path = PROJECT_ROOT / configured
    if not path.is_file():
        readiness_rows.append({'model_key': model_key, 'reporting_summary': str(configured), 'status': 'MISSING'})
        continue
    summary = validate_reporting_directory(path.parent)
    summaries.append(summary)
    readiness_rows.append({'model_key': model_key, 'reporting_summary': str(configured), 'status': summary.get('reporting_status', 'INVALID')})

readiness = pd.DataFrame(readiness_rows)
display(readiness)

,model_key,reporting_summary,status
0,lr,NOT_CONFIGURED,PENDING
1,rf,artifacts/dataset_b/rf/final/rf_final_15f_2026...,COMPLETE
2,xgboost,artifacts/dataset_b/xgboost/final/xgb_final_15...,COMPLETE
3,plain_mlp,artifacts/dataset_b/plain_mlp/final/plain_mlp_...,COMPLETE
4,realmlp,NOT_CONFIGURED,PENDING
5,tabmmini,NOT_CONFIGURED,PENDING


## Results

Cross-model tables và charts chỉ được mở sau khi gate dưới đây PASS.

In [3]:
comparison_ready = False
if len(summaries) == len(FINAL_MODEL_KEYS):
    validate_cross_model_summaries(summaries)
    comparison_ready = True

if comparison_ready:
    display(Markdown('**PASS:** Sáu final reporting packages đã sẵn sàng cho tổng hợp.'))
else:
    display(Markdown(f'**PENDING:** mới có {len(summaries)}/{len(FINAL_MODEL_KEYS)} reporting summaries được cấu hình; chưa mở so sánh.'))

**PENDING:** mới có 3/6 reporting summaries được cấu hình; chưa mở so sánh.

### Visual contract sau khi gate PASS

1. CV ROC-AUC: fold dot plot theo model, cùng scale.
2. RQ1 metrics: small-multiple forest plots với point estimate và CI 95%.
3. Calibration: small multiples cùng trục và đường lý tưởng.
4. Confusion/recall: bảng và heatmap chuẩn hóa.
5. Threshold và timing: bảng; chỉ so sánh timing khi định nghĩa tương đương.

Không dùng radar chart, không xếp hạng bằng metric không cùng hướng và không suy ra statistical significance từ CI riêng lẻ.

## Takeaways

Scaffold đã khóa đầu vào và gate so sánh. Các chart tổng hợp sẽ được nối vào cùng notebook khi sáu model có reporting package hoàn chỉnh; trước thời điểm đó trạng thái đúng là `PENDING`.